# Train Faster R-CNN, EfficientDet, RT-DETR

This notebook extends the YOLO comparison in `shrimp.ipynb` with three more detectors.

Fair-comparison rules used here:

- Same train/valid/test split as YOLO, exported from the same Roboflow project.
- Same image size: `512`.
- Same seeds: `0, 1, 2`.
- Same maximum epochs: `150`.
- Best checkpoint is selected by validation `mAP50-95`, not by train loss.
- Test split is evaluated only after checkpoint selection.
- Results are reported per run and as mean/std over seeds.

Use COCO JSON export for Faster R-CNN and EfficientDet, and YOLO export for RT-DETR through Ultralytics.

## Install Dependencies

In [1]:
!unzip /content/data.zip -d /content/data

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  inflating: /content/data/data/tom_benh.v1i.coco/train/76_jpg.rf.58a0a484913f3dd9495998301c327930.jpg  
  inflating: /content/data/__MACOSX/data/tom_benh.v1i.coco/train/._76_jpg.rf.58a0a484913f3dd9495998301c327930.jpg  
  inflating: /content/data/data/tom_benh.v1i.coco/train/26_jpg.rf.4ef707c555ca4b6199b13755358bbc9f.jpg  
  inflating: /content/data/__MACOSX/data/tom_benh.v1i.coco/train/._26_jpg.rf.4ef707c555ca4b6199b13755358bbc9f.jpg  
  inflating: /content/data/data/tom_benh.v1i.coco/train/88_jpg.rf.1fc77517a3e4c225ba2bc26669b6053d.jpg  
  inflating: /content/data/__MACOSX/data/tom_benh.v1i.coco/train/._88_jpg.rf.1fc77517a3e4c225ba2bc26669b6053d.jpg  
  inflating: /content/data/data/tom_benh.v1i.coco/train/58_jpg.rf.a4e9b93bcdbdd1904c83fafc1febc64c.jpg  
  inflating: /content/data/__MACOSX/data/tom_benh.v1i.coco/train/._58_jpg.rf.a4e9b93bcdbdd1904c83fafc1febc64c.jpg  
  inflating: /content/data/data/tom_benh.v1i.coco/train/121

In [2]:
!pip install -q ultralytics effdet pycocotools torchmetrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.9 MB/s eta 0:00:00


## Configuration

In [8]:
from pathlib import Path
from typing import Optional
import json
import math
import random
import time

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as TF
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.ops import box_iou
from pycocotools.coco import COCO
from torchmetrics.detection.mean_ap import MeanAveragePrecision

from pathlib import Path

# Thư mục dữ liệu gốc sau khi giải nén
DATA_ROOT = Path('/content/data/data')

# COCO JSON export dùng cho Faster R-CNN và EfficientDet
COCO_ROOT = DATA_ROOT / 'tom_benh.v1i.coco'

COCO_TRAIN_JSON = COCO_ROOT / 'train' / '_annotations.coco.json'
COCO_VAL_JSON = COCO_ROOT / 'valid' / '_annotations.coco.json'
COCO_TEST_JSON = COCO_ROOT / 'test' / '_annotations.coco.json'

# YOLOv11 export dùng cho RT-DETR thông qua Ultralytics
YOLO_ROOT = DATA_ROOT / 'tom_benh.v1i.yolov11'
YOLO_DATA_YAML = YOLO_ROOT / 'data.yaml'

# Thư mục lưu kết quả huấn luyện trên Google Drive
OUTPUT_ROOT = Path('/content/drive/MyDrive/shrimp/runs_detection_models')
YOLO_RUNS_ROOT = Path('/content/drive/MyDrive/shrimp/runs')
YOLO_SUMMARY_PER_RUN = Path(
    '/content/drive/MyDrive/shrimp/test_summary_per_run.csv'
)
IMG_SIZE = 512
EPOCHS = 150
PATIENCE = 20
SEEDS = [0, 1, 2]

# YOLO was trained with batch=32. The custom PyTorch detectors usually cannot fit
# batch=32 on a Colab T4, so we use gradient accumulation to keep an effective
# batch size close to YOLO for optimization.
BATCH_SIZE = 4
EFFECTIVE_BATCH_SIZE = 32
ACCUM_STEPS = max(1, math.ceil(EFFECTIVE_BATCH_SIZE / BATCH_SIZE))

NUM_WORKERS = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SCORE_THRESHOLD = 0.001

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('device:', DEVICE)
print('coco root:', COCO_ROOT)
print('yolo data yaml:', YOLO_DATA_YAML)
print('output root:', OUTPUT_ROOT)
print('img size:', IMG_SIZE)
print('epochs:', EPOCHS)
print('patience:', PATIENCE)
print('batch size:', BATCH_SIZE)
print('effective batch size:', BATCH_SIZE * ACCUM_STEPS)
print('accum steps:', ACCUM_STEPS)

device: cuda
coco root: /content/data/data/tom_benh.v1i.coco
yolo data yaml: /content/data/data/tom_benh.v1i.yolov11/data.yaml
output root: /content/drive/MyDrive/shrimp/runs_detection_models
img size: 512
epochs: 150
patience: 20
batch size: 4
effective batch size: 32
accum steps: 8


In [9]:
paths_to_check = {
    'COCO train JSON': COCO_TRAIN_JSON,
    'COCO validation JSON': COCO_VAL_JSON,
    'COCO test JSON': COCO_TEST_JSON,
    'YOLO data.yaml': YOLO_DATA_YAML,
}

for name, path in paths_to_check.items():
    print(f'{name}:')
    print(f'  Path: {path}')
    print(f'  Exists: {path.exists()}')

COCO train JSON:
  Path: /content/data/data/tom_benh.v1i.coco/train/_annotations.coco.json
  Exists: True
COCO validation JSON:
  Path: /content/data/data/tom_benh.v1i.coco/valid/_annotations.coco.json
  Exists: True
COCO test JSON:
  Path: /content/data/data/tom_benh.v1i.coco/test/_annotations.coco.json
  Exists: True
YOLO data.yaml:
  Path: /content/data/data/tom_benh.v1i.yolov11/data.yaml
  Exists: True


## Dataset And Reproducibility Utilities

In [10]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def require_files(paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        message = 'Missing required dataset files:' + chr(10) + chr(10).join(missing)
        raise FileNotFoundError(message)


def letterbox_image_and_boxes(image: Image.Image, boxes_xywh, img_size: int):
    """YOLO-style letterbox resize that preserves aspect ratio."""
    orig_w, orig_h = image.size
    scale = min(img_size / orig_w, img_size / orig_h)
    new_w = int(round(orig_w * scale))
    new_h = int(round(orig_h * scale))
    pad_x = (img_size - new_w) / 2.0
    pad_y = (img_size - new_h) / 2.0

    resized = image.resize((new_w, new_h), Image.BILINEAR)
    canvas = Image.new('RGB', (img_size, img_size), (114, 114, 114))
    canvas.paste(resized, (int(round(pad_x)), int(round(pad_y))))

    boxes = []
    for x, y, w, h in boxes_xywh:
        x1 = x * scale + pad_x
        y1 = y * scale + pad_y
        x2 = (x + w) * scale + pad_x
        y2 = (y + h) * scale + pad_y
        x1 = max(0.0, min(float(img_size - 1), x1))
        y1 = max(0.0, min(float(img_size - 1), y1))
        x2 = max(0.0, min(float(img_size - 1), x2))
        y2 = max(0.0, min(float(img_size - 1), y2))
        boxes.append([x1, y1, x2, y2])
    return canvas, boxes


class CocoDetectionDataset(Dataset):
    def __init__(self, annotation_file: Path, img_size: int = 512, effdet_targets: bool = False):
        self.annotation_file = Path(annotation_file)
        self.image_dir = self.annotation_file.parent
        self.img_size = img_size
        self.effdet_targets = effdet_targets
        self.coco = COCO(str(self.annotation_file))
        self.image_ids = sorted(self.coco.getImgIds())
        self.cat_ids = sorted(self.coco.getCatIds())
        self.cat_id_to_label = {cat_id: idx + 1 for idx, cat_id in enumerate(self.cat_ids)}
        self.label_to_cat_id = {label: cat_id for cat_id, label in self.cat_id_to_label.items()}
        self.class_names = [self.coco.cats[cat_id]['name'] for cat_id in self.cat_ids]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.coco.loadImgs(image_id)[0]
        image_path = self.image_dir / image_info['file_name']
        image = Image.open(image_path).convert('RGB')

        ann_ids = self.coco.getAnnIds(imgIds=image_id, iscrowd=None)
        anns = self.coco.loadAnns(ann_ids)

        raw_boxes = []
        labels = []
        iscrowd = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            if w <= 0 or h <= 0:
                continue
            raw_boxes.append([x, y, w, h])
            labels.append(self.cat_id_to_label[ann['category_id']])
            iscrowd.append(int(ann.get('iscrowd', 0)))

        image, boxes = letterbox_image_and_boxes(image, raw_boxes, self.img_size)
        valid_boxes = []
        valid_labels = []
        valid_iscrowd = []
        areas = []
        for box, label, crowd in zip(boxes, labels, iscrowd):
            x1, y1, x2, y2 = box
            if x2 <= x1 or y2 <= y1:
                continue
            valid_boxes.append(box)
            valid_labels.append(label)
            valid_iscrowd.append(crowd)
            areas.append((x2 - x1) * (y2 - y1))

        image_tensor = TF.to_tensor(image)
        boxes_tensor = torch.tensor(valid_boxes, dtype=torch.float32).reshape(-1, 4)
        labels_tensor = torch.tensor(valid_labels, dtype=torch.int64)

        target = {
            'boxes': boxes_tensor,
            'labels': labels_tensor,
            'image_id': torch.tensor([image_id], dtype=torch.int64),
            'area': torch.tensor(areas, dtype=torch.float32),
            'iscrowd': torch.tensor(valid_iscrowd, dtype=torch.int64),
        }

        if not self.effdet_targets:
            return image_tensor, target

        # effdet training uses yxyx boxes and one-based class ids; -1 pads ignored boxes.
        boxes_yxyx = boxes_tensor[:, [1, 0, 3, 2]] if len(boxes_tensor) else torch.zeros((0, 4), dtype=torch.float32)
        return image_tensor, {'bbox': boxes_yxyx, 'cls': labels_tensor}


def detection_collate(batch):
    return tuple(zip(*batch))


def efficientdet_collate(batch):
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)

    max_boxes = max(t['bbox'].shape[0] for t in targets)
    max_boxes = max(max_boxes, 1)

    bbox = torch.zeros((len(targets), max_boxes, 4), dtype=torch.float32)
    cls = torch.full((len(targets), max_boxes), -1, dtype=torch.int64)
    img_scale = torch.ones((len(targets),), dtype=torch.float32)
    img_size = torch.full((len(targets), 2), IMG_SIZE, dtype=torch.float32)

    for i, t in enumerate(targets):
        n = t['bbox'].shape[0]
        if n:
            bbox[i, :n] = t['bbox']
            cls[i, :n] = t['cls']

    return images, {'bbox': bbox, 'cls': cls, 'img_scale': img_scale, 'img_size': img_size}


def make_detection_loader(ds, batch_size: int, shuffle: bool, seed: Optional[int] = None):
    generator = None
    if seed is not None:
        generator = torch.Generator()
        generator.manual_seed(seed)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        collate_fn=detection_collate,
        generator=generator,
    )


def make_effdet_loader(ds, batch_size: int, shuffle: bool, seed: Optional[int] = None):
    generator = None
    if seed is not None:
        generator = torch.Generator()
        generator.manual_seed(seed)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        collate_fn=efficientdet_collate,
        generator=generator,
    )


require_files([COCO_TRAIN_JSON, COCO_VAL_JSON, COCO_TEST_JSON, YOLO_DATA_YAML])

train_ds = CocoDetectionDataset(COCO_TRAIN_JSON, IMG_SIZE)
val_ds = CocoDetectionDataset(COCO_VAL_JSON, IMG_SIZE)
test_ds = CocoDetectionDataset(COCO_TEST_JSON, IMG_SIZE)

eff_train_ds = CocoDetectionDataset(COCO_TRAIN_JSON, IMG_SIZE, effdet_targets=True)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)

print('classes:', CLASS_NAMES)
print('num classes:', NUM_CLASSES)
print('train images:', len(train_ds), 'val images:', len(val_ds), 'test images:', len(test_ds))

loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
classes: ['shrimp', '0', '1']
num classes: 3
train images: 707 val images: 202 test images: 101


## Shared Evaluation Helpers

In [11]:
def new_pr_counts():
    return {'tp': 0, 'fp': 0, 'fn': 0}


def update_pr_counts(preds, targets, counts, iou_threshold=0.5):
    for pred, target in zip(preds, targets):
        pred_boxes = pred['boxes']
        pred_scores = pred['scores']
        pred_labels = pred['labels']
        target_boxes = target['boxes']
        target_labels = target['labels']

        if len(pred_boxes):
            order = torch.argsort(pred_scores, descending=True)
            pred_boxes = pred_boxes[order]
            pred_labels = pred_labels[order]

        matched_targets = set()
        for p_box, p_label in zip(pred_boxes, pred_labels):
            candidate_idx = [
                i for i, t_label in enumerate(target_labels)
                if i not in matched_targets and int(t_label) == int(p_label)
            ]
            if not candidate_idx:
                counts['fp'] += 1
                continue

            candidate_boxes = target_boxes[candidate_idx]
            ious = box_iou(p_box.reshape(1, 4), candidate_boxes).reshape(-1)
            best_pos = int(torch.argmax(ious).item())
            if float(ious[best_pos]) >= iou_threshold:
                counts['tp'] += 1
                matched_targets.add(candidate_idx[best_pos])
            else:
                counts['fp'] += 1

        counts['fn'] += max(0, len(target_boxes) - len(matched_targets))


def precision_recall_from_counts(counts):
    tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return precision, recall


def metric_to_row(metric_result: dict, pr_counts: dict):
    precision, recall = precision_recall_from_counts(pr_counts)
    return {
        'precision': float(precision),
        'recall': float(recall),
        'mAP50': float(metric_result['map_50'].item()),
        'mAP50-95': float(metric_result['map'].item()),
    }


@torch.no_grad()
def evaluate_torchvision_detector(model, data_loader, device=DEVICE, score_threshold=SCORE_THRESHOLD):
    model.eval()
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    pr_counts = new_pr_counts()
    total_infer_ms = []

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        metric_targets = [
            {'boxes': t['boxes'].cpu(), 'labels': t['labels'].cpu()}
            for t in targets
        ]

        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(images)
        if device == 'cuda':
            torch.cuda.synchronize()
        total_infer_ms.append((time.perf_counter() - t0) * 1000.0 / max(1, len(images)))

        preds = []
        for out in outputs:
            scores = out['scores'].detach().cpu()
            keep = scores >= score_threshold
            preds.append({
                'boxes': out['boxes'].detach().cpu()[keep],
                'scores': scores[keep],
                'labels': out['labels'].detach().cpu()[keep],
            })
        metric.update(preds, metric_targets)
        update_pr_counts(preds, metric_targets, pr_counts, iou_threshold=0.5)

    result = metric.compute()
    row = metric_to_row(result, pr_counts)
    row['inference_ms'] = float(np.mean(total_infer_ms)) if total_infer_ms else np.nan
    return row


def save_json(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding='utf-8')


def save_run_result(run_dir: Path, row: dict):
    pd.DataFrame([row]).to_csv(run_dir / 'test_metrics.csv', index=False)
    save_json(run_dir / 'test_metrics.json', row)


def summarize_results(rows, output_root=OUTPUT_ROOT):
    df_runs = pd.DataFrame(rows)
    per_run_path = output_root / 'detection_models_test_summary_per_run.csv'
    summary_path = output_root / 'detection_models_test_summary_mean_std.csv'
    df_runs.to_csv(per_run_path, index=False)

    metric_cols = [
        'precision', 'recall', 'mAP50', 'mAP50-95',
        'inference_ms', 'best_epoch', 'best_val_mAP50-95'
    ]
    metric_cols = [c for c in metric_cols if c in df_runs.columns]
    df_summary = df_runs.groupby('model')[metric_cols].agg(['mean', 'std'])
    df_summary.columns = [f'{col}_{stat}' for col, stat in df_summary.columns]
    df_summary = df_summary.reset_index()
    df_summary.to_csv(summary_path, index=False)

    print('per-run saved:', per_run_path)
    print('summary saved:', summary_path)
    display(df_runs)
    display(df_summary)
    return df_runs, df_summary

## Faster R-CNN Training And Evaluation

In [12]:
def build_faster_rcnn(num_classes: int):
    # num_classes includes the background class.
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
        weights=weights,
        min_size=IMG_SIZE,
        max_size=IMG_SIZE,
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def train_faster_rcnn(seed: int):
    set_seed(seed)
    run_name = f'fasterrcnn_resnet50_fpn_v2_seed{seed}'
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    train_loader = make_detection_loader(train_ds, BATCH_SIZE, shuffle=True, seed=seed)
    val_loader = make_detection_loader(val_ds, BATCH_SIZE, shuffle=False)
    test_loader = make_detection_loader(test_ds, BATCH_SIZE, shuffle=False)

    model = build_faster_rcnn(NUM_CLASSES + 1).to(DEVICE)
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=0.005,
        momentum=0.9,
        weight_decay=0.0005,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=45, gamma=0.1)

    history = []
    best_val_map = -1.0
    best_epoch = -1
    epochs_without_improvement = 0

    optimizer.zero_grad(set_to_none=True)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for step, (images, targets) in enumerate(train_loader, start=1):
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            (loss / ACCUM_STEPS).backward()

            if step % ACCUM_STEPS == 0 or step == len(train_loader):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            losses.append(float(loss.detach().cpu()))

        scheduler.step()
        val_metrics = evaluate_torchvision_detector(model, val_loader)
        mean_loss = float(np.mean(losses))
        val_map = val_metrics['mAP50-95']
        history_row = {
            'epoch': epoch,
            'train_loss': mean_loss,
            **{f'val_{k}': v for k, v in val_metrics.items()},
        }
        history.append(history_row)
        print(f'[{run_name}] epoch {epoch:03d}/{EPOCHS} train_loss={mean_loss:.4f} val_mAP50-95={val_map:.4f} val_mAP50={val_metrics["mAP50"]:.4f}')

        if val_map > best_val_map:
            best_val_map = val_map
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save({
                'model_state': model.state_dict(),
                'class_names': CLASS_NAMES,
                'img_size': IMG_SIZE,
                'seed': seed,
                'epoch': epoch,
                'best_val_mAP50-95': best_val_map,
            }, run_dir / 'best.pt')
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(f'[{run_name}] early stopping at epoch {epoch}; best_epoch={best_epoch} best_val_mAP50-95={best_val_map:.4f}')
            break

    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)

    checkpoint = torch.load(run_dir / 'best.pt', map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    test_metrics = evaluate_torchvision_detector(model, test_loader)
    row = {
        'model': 'Faster R-CNN ResNet50 FPN v2',
        'seed': seed,
        'run': run_name,
        'best_epoch': best_epoch,
        'best_val_mAP50-95': best_val_map,
        **test_metrics,
    }
    save_run_result(run_dir, row)
    return row


faster_rcnn_rows = []
for seed in SEEDS:
    faster_rcnn_rows.append(train_faster_rcnn(seed))

pd.DataFrame(faster_rcnn_rows)

[fasterrcnn_resnet50_fpn_v2_seed0] epoch 001/150 train_loss=0.9209 val_mAP50-95=0.1473 val_mAP50=0.4140
[fasterrcnn_resnet50_fpn_v2_seed0] epoch 002/150 train_loss=0.5178 val_mAP50-95=0.3482 val_mAP50=0.5797
[fasterrcnn_resnet50_fpn_v2_seed0] epoch 003/150 train_loss=0.3428 val_mAP50-95=0.4579 val_mAP50=0.6342
[fasterrcnn_resnet50_fpn_v2_seed0] epoch 004/150 train_loss=0.2834 val_mAP50-95=0.5206 val_mAP50=0.7067
[fasterrcnn_resnet50_fpn_v2_seed0] epoch 005/150 train_loss=0.2464 val_mAP50-95=0.5931 val_mAP50=0.7808
[fasterrcnn_resnet50_fpn_v2_seed0] epoch 006/150 train_loss=0.2158 val_mAP50-95=0.6443 val_mAP50=0.8308
[fasterrcnn_resnet50_fpn_v2_seed0] epoch 007/150 train_loss=0.1850 val_mAP50-95=0.6953 val_mAP50=0.8751
[fasterrcnn_resnet50_fpn_v2_seed0] epoch 008/150 train_loss=0.1644 val_mAP50-95=0.7076 val_mAP50=0.8929
[fasterrcnn_resnet50_fpn_v2_seed0] epoch 009/150 train_loss=0.1394 val_mAP50-95=0.7320 val_mAP50=0.9028
[fasterrcnn_resnet50_fpn_v2_seed0] epoch 010/150 train_loss=0.12

,model,seed,run,best_epoch,best_val_mAP50-95,precision,recall,mAP50,mAP50-95,inference_ms
0,Faster R-CNN ResNet50 FPN v2,0,fasterrcnn_resnet50_fpn_v2_seed0,56,0.760215,0.564639,0.928125,0.874963,0.707457,16.991587
1,Faster R-CNN ResNet50 FPN v2,1,fasterrcnn_resnet50_fpn_v2_seed1,51,0.766708,0.610063,0.909375,0.860846,0.691223,13.431556
2,Faster R-CNN ResNet50 FPN v2,2,fasterrcnn_resnet50_fpn_v2_seed2,83,0.769670,0.601240,0.909375,0.858685,0.690967,13.811558


## EfficientDet Training And Evaluation

In [15]:
from effdet import create_model


def build_efficientdet_train(num_classes: int):
    return create_model(
        'tf_efficientdet_d0',
        bench_task='train',
        bench_labeler=True,
        num_classes=num_classes,
        pretrained=True,
        image_size=(IMG_SIZE, IMG_SIZE),
    )


def build_efficientdet_predict(num_classes: int):
    return create_model(
        'tf_efficientdet_d0',
        bench_task='predict',
        num_classes=num_classes,
        pretrained=False,
        image_size=(IMG_SIZE, IMG_SIZE),
    )


def parse_effdet_detections(detections: torch.Tensor):
    # effdet predict bench returns detections shaped [B, N, 6]: x1, y1, x2, y2, score, class.
    preds = []
    detections = detections.detach().cpu()
    for det in detections:
        if det.numel() == 0:
            preds.append({
                'boxes': torch.zeros((0, 4), dtype=torch.float32),
                'scores': torch.zeros((0,), dtype=torch.float32),
                'labels': torch.zeros((0,), dtype=torch.int64),
            })
            continue
        scores = det[:, 4]
        keep = scores >= SCORE_THRESHOLD
        labels = det[:, 5].to(torch.int64)
        labels = torch.clamp(labels, min=1, max=NUM_CLASSES)
        preds.append({
            'boxes': det[:, :4][keep].to(torch.float32),
            'scores': scores[keep].to(torch.float32),
            'labels': labels[keep],
        })
    return preds


@torch.no_grad()
def evaluate_efficientdet_from_checkpoint(checkpoint_path: Path, data_loader, device=DEVICE):
    model = build_efficientdet_predict(NUM_CLASSES).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state'], strict=False)
    model.eval()

    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    pr_counts = new_pr_counts()
    total_infer_ms = []

    for images, targets in data_loader:
        images = torch.stack([img for img in images], dim=0).to(device)
        img_info = {
            'img_scale': torch.ones((images.shape[0],), dtype=torch.float32, device=device),
            'img_size': torch.full((images.shape[0], 2), IMG_SIZE, dtype=torch.float32, device=device),
        }
        metric_targets = [
            {'boxes': t['boxes'].cpu(), 'labels': t['labels'].cpu()}
            for t in targets
        ]

        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        detections = model(images, img_info)
        if device == 'cuda':
            torch.cuda.synchronize()
        total_infer_ms.append((time.perf_counter() - t0) * 1000.0 / max(1, images.shape[0]))

        if isinstance(detections, (tuple, list)):
            detections = detections[0]
        preds = parse_effdet_detections(detections)
        metric.update(preds, metric_targets)
        update_pr_counts(preds, metric_targets, pr_counts, iou_threshold=0.5)

    result = metric.compute()
    row = metric_to_row(result, pr_counts)
    row['inference_ms'] = float(np.mean(total_infer_ms)) if total_infer_ms else np.nan
    return row


def train_efficientdet(seed: int):
    set_seed(seed)
    run_name = f'efficientdet_d0_seed{seed}'
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    train_loader = make_effdet_loader(eff_train_ds, BATCH_SIZE, shuffle=True, seed=seed)
    val_loader = make_detection_loader(val_ds, BATCH_SIZE, shuffle=False)
    test_loader = make_detection_loader(test_ds, BATCH_SIZE, shuffle=False)

    model = build_efficientdet_train(NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = []
    best_val_map = -1.0
    best_epoch = -1
    epochs_without_improvement = 0
    tmp_checkpoint = run_dir / '_epoch_tmp.pt'

    optimizer.zero_grad(set_to_none=True)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []

        for step, (images, targets) in enumerate(train_loader, start=1):
            images = images.to(DEVICE)
            targets = {k: v.to(DEVICE) for k, v in targets.items()}

            loss_dict = model(images, targets)
            loss = loss_dict['loss'] if isinstance(loss_dict, dict) else loss_dict
            (loss / ACCUM_STEPS).backward()

            if step % ACCUM_STEPS == 0 or step == len(train_loader):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            losses.append(float(loss.detach().cpu()))

        scheduler.step()
        torch.save({
            'model_state': model.state_dict(),
            'class_names': CLASS_NAMES,
            'img_size': IMG_SIZE,
            'seed': seed,
            'epoch': epoch,
        }, tmp_checkpoint)
        val_metrics = evaluate_efficientdet_from_checkpoint(tmp_checkpoint, val_loader)
        mean_loss = float(np.mean(losses))
        val_map = val_metrics['mAP50-95']
        history_row = {
            'epoch': epoch,
            'train_loss': mean_loss,
            **{f'val_{k}': v for k, v in val_metrics.items()},
        }
        history.append(history_row)
        print(f'[{run_name}] epoch {epoch:03d}/{EPOCHS} train_loss={mean_loss:.4f} val_mAP50-95={val_map:.4f} val_mAP50={val_metrics["mAP50"]:.4f}')

        if val_map > best_val_map:
            best_val_map = val_map
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save({
                'model_state': model.state_dict(),
                'class_names': CLASS_NAMES,
                'img_size': IMG_SIZE,
                'seed': seed,
                'epoch': epoch,
                'best_val_mAP50-95': best_val_map,
            }, run_dir / 'best.pt')
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(f'[{run_name}] early stopping at epoch {epoch}; best_epoch={best_epoch} best_val_mAP50-95={best_val_map:.4f}')
            break

    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    tmp_checkpoint.unlink(missing_ok=True)

    test_metrics = evaluate_efficientdet_from_checkpoint(run_dir / 'best.pt', test_loader)
    row = {
        'model': 'EfficientDet-D0',
        'seed': seed,
        'run': run_name,
        'best_epoch': best_epoch,
        'best_val_mAP50-95': best_val_map,
        **test_metrics,
    }
    save_run_result(run_dir, row)
    return row


efficientdet_rows = []
for seed in SEEDS:
    efficientdet_rows.append(train_efficientdet(seed))

pd.DataFrame(efficientdet_rows)

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

[efficientdet_d0_seed0] epoch 001/150 train_loss=1.3989 val_mAP50-95=0.0652 val_mAP50=0.1492


[efficientdet_d0_seed0] epoch 002/150 train_loss=1.1487 val_mAP50-95=0.1817 val_mAP50=0.3280


[efficientdet_d0_seed0] epoch 003/150 train_loss=1.0019 val_mAP50-95=0.2503 val_mAP50=0.4176


...

[efficientdet_d0_seed2] epoch 080/150 train_loss=0.0998 val_mAP50-95=0.7209 val_mAP50=0.9257


[efficientdet_d0_seed2] epoch 081/150 train_loss=0.1006 val_mAP50-95=0.7177 val_mAP50=0.9249


[efficientdet_d0_seed2] epoch 082/150 train_loss=0.0970 val_mAP50-95=0.7199 val_mAP50=0.9243


[efficientdet_d0_seed2] epoch 083/150 train_loss=0.0978 val_mAP50-95=0.7185 val_mAP50=0.9220
[efficientdet_d0_seed2] early stopping at epoch 83; best_epoch=63 best_val_mAP50-95=0.7214


,model,seed,run,best_epoch,best_val_mAP50-95,precision,recall,mAP50,mAP50-95,inference_ms
0,EfficientDet-D0,0,efficientdet_d0_seed0,100,0.731596,0.030990,0.978125,0.900497,0.692968,15.882019
1,EfficientDet-D0,1,efficientdet_d0_seed1,69,0.720706,0.030891,0.975000,0.888401,0.671769,10.755861
2,EfficientDet-D0,2,efficientdet_d0_seed2,63,0.721444,0.030891,0.975000,0.898287,0.690050,10.882426


## RT-DETR Training And Evaluation

In [16]:
from ultralytics import RTDETR


def train_rtdetr(seed: int):
    set_seed(seed)
    run_name = f'rtdetr_l_seed{seed}'
    print(f'========== TRAIN RT-DETR | seed={seed} | run={run_name} ==========')

    model = RTDETR('rtdetr-l.pt')
    model.train(
        data=str(YOLO_DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        nbs=EFFECTIVE_BATCH_SIZE,
        patience=20,
        project=str(OUTPUT_ROOT),
        name=run_name,
        device=0 if DEVICE == 'cuda' else 'cpu',
        seed=seed,
        deterministic=True,
    )

    model_path = OUTPUT_ROOT / run_name / 'weights' / 'best.pt'
    model = RTDETR(str(model_path))
    metrics = model.val(
        data=str(YOLO_DATA_YAML),
        split='test',
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=0 if DEVICE == 'cuda' else 'cpu',
        verbose=False,
    )
    row = {
        'model': 'RT-DETR-L',
        'seed': seed,
        'run': run_name,
        'best_epoch': np.nan,
        'best_val_mAP50-95': np.nan,
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
        'preprocess_ms': float(metrics.speed.get('preprocess', 0.0)),
        'inference_ms': float(metrics.speed.get('inference', 0.0)),
        'postprocess_ms': float(metrics.speed.get('postprocess', 0.0)),
    }
    save_run_result(OUTPUT_ROOT / run_name, row)
    return row


rtdetr_rows = []
for seed in SEEDS:
    rtdetr_rows.append(train_rtdetr(seed))

pd.DataFrame(rtdetr_rows)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
========== TRAIN RT-DETR | seed=0 | run=rtdetr_l_seed0 ==========
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/data/tom_benh.v1i.yolov11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, e

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/150      6.56G      0.558     0.8893     0.4512         27        512: 100% ━━━━━━━━━━━━ 177/177 2.8it/s 1:03
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 4.9it/s 5.3s
                   all        202        660      0.818      0.747      0.829      0.682

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/150      6.68G     0.5575     0.3754     0.3454         21        512: 0% ──────────── 0/177  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/150      6.68G     0.3485     0.5823      0.228         21        512: 100% ━━━━━━━━━━━━ 177/177 3.4it/s 52.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 11.0it/s 2.4s
                   all        202        660      0.763      0.711      0.783      0.601

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/150      6.68G     0.2968     0.5285     0.1897         22        512: 0% ──────────── 0/177  0.3s

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    125/150      5.31G     0.1379     0.2614    0.07903         16        512: 100% ━━━━━━━━━━━━ 177/177 3.3it/s 53.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 10.8it/s 2.4s
                   all        202        660      0.919      0.904      0.939      0.845

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    126/150      5.31G     0.1709     0.2954     0.0436         18        512: 0% ──────────── 0/177  0.3s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    126/150      5.31G     0.1373     0.2628    0.07917         29        512: 100% ━━━━━━━━━━━━ 177/177 3.3it/s 53.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 11.0it/s 2.4s
                   all        202        660      0.902      0.917      0.943      0.848
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 106, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

126 epochs completed in 2.007 hours.
Optimizer stripped from /content/drive/MyDrive/shrimp/runs_detection_models/rtdetr_l_seed2/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/shrimp/runs_detection_models/rtdetr_l_seed2/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/shrimp/runs_detection_models/rtdetr_l_seed2/weights/best.pt...
Ultralytics 8.4.104

,model,seed,run,best_epoch,best_val_mAP50-95,precision,recall,mAP50,mAP50-95,preprocess_ms,inference_ms,postprocess_ms
0,RT-DETR-L,0,rtdetr_l_seed0,NaN,NaN,0.882541,0.875722,0.911389,0.800865,0.522928,12.085252,0.196965
1,RT-DETR-L,1,rtdetr_l_seed1,NaN,NaN,0.861996,0.842680,0.880085,0.759624,0.558458,11.779083,0.202699
2,RT-DETR-L,2,rtdetr_l_seed2,NaN,NaN,0.885382,0.865639,0.912915,0.794719,1.141763,11.526405,0.207628


## Save Detection-Model Summary

In [17]:
all_detection_rows = []
for name in ['faster_rcnn_rows', 'efficientdet_rows', 'rtdetr_rows']:
    all_detection_rows.extend(globals().get(name, []))

detection_runs_df, detection_summary_df = summarize_results(all_detection_rows, OUTPUT_ROOT)

per-run saved: /content/drive/MyDrive/shrimp/runs_detection_models/detection_models_test_summary_per_run.csv
summary saved: /content/drive/MyDrive/shrimp/runs_detection_models/detection_models_test_summary_mean_std.csv


,model,seed,run,best_epoch,best_val_mAP50-95,precision,recall,mAP50,mAP50-95,inference_ms,preprocess_ms,postprocess_ms
0,Faster R-CNN ResNet50 FPN v2,0,fasterrcnn_resnet50_fpn_v2_seed0,56.0,0.760215,0.564639,0.928125,0.874963,0.707457,16.991587,NaN,NaN
1,Faster R-CNN ResNet50 FPN v2,1,fasterrcnn_resnet50_fpn_v2_seed1,51.0,0.766708,0.610063,0.909375,0.860846,0.691223,13.431556,NaN,NaN
2,Faster R-CNN ResNet50 FPN v2,2,fasterrcnn_resnet50_fpn_v2_seed2,83.0,0.769670,0.601240,0.909375,0.858685,0.690967,13.811558,NaN,NaN
3,EfficientDet-D0,0,efficientdet_d0_seed0,100.0,0.731596,0.030990,0.978125,0.900497,0.692968,15.882019,NaN,NaN
4,EfficientDet-D0,1,efficientdet_d0_seed1,69.0,0.720706,0.030891,0.975000,0.888401,0.671769,10.755861,NaN,NaN
5,EfficientDet-D0,2,efficientdet_d0_seed2,63.0,0.721444,0.030891,0.975000,0.898287,0.690050,10.882426,NaN,NaN
6,RT-DETR-L,0,rtdetr_l_seed0,NaN,NaN,0.882541,0.875722,0.911389,0.800865,12.085252,0.522928,0.196965
7,RT-DETR-L,1,rtdetr_l_seed1,NaN,NaN,0.861996,0.842680,0.880085,0.759624,11.779083,0.558458,0.202699
8,RT-DETR-L,2,rtdetr_l_seed2,NaN,NaN,0.885382,0.865639,0.912915,0.794719,11.526405,1.141763,0.207628


,model,precision_mean,precision_std,recall_mean,recall_std,mAP50_mean,mAP50_std,mAP50-95_mean,mAP50-95_std,inference_ms_mean,inference_ms_std,best_epoch_mean,best_epoch_std,best_val_mAP50-95_mean,best_val_mAP50-95_std
0,EfficientDet-D0,0.030924,0.000057,0.976042,0.001804,0.895728,0.006441,0.684929,0.011490,12.506769,2.923737,77.333333,19.857828,0.724582,0.006085
1,Faster R-CNN ResNet50 FPN v2,0.591980,0.024086,0.915625,0.010825,0.864831,0.008840,0.696549,0.009447,14.744900,1.954942,63.333333,17.214335,0.765531,0.004836
2,RT-DETR-L,0.876640,0.012761,0.861347,0.016934,0.901463,0.018530,0.785069,0.022250,11.796913,0.279850,NaN,NaN,NaN,NaN


## Optional: Combine With Existing YOLO Summary

In [18]:
if YOLO_SUMMARY_PER_RUN.exists():
    yolo_df = pd.read_csv(YOLO_SUMMARY_PER_RUN)
    detection_df = pd.read_csv(OUTPUT_ROOT / 'detection_models_test_summary_per_run.csv')
    common_cols = sorted(set(yolo_df.columns).union(detection_df.columns))
    combined = pd.concat([
        yolo_df.reindex(columns=common_cols),
        detection_df.reindex(columns=common_cols),
    ], ignore_index=True)

    combined_path = OUTPUT_ROOT / 'all_models_test_summary_per_run.csv'
    combined_summary_path = OUTPUT_ROOT / 'all_models_test_summary_mean_std.csv'
    combined.to_csv(combined_path, index=False)

    metric_cols = [
        'precision', 'recall', 'mAP50', 'mAP50-95',
        'preprocess_ms', 'inference_ms', 'postprocess_ms',
        'best_epoch', 'best_val_mAP50-95'
    ]
    metric_cols = [c for c in metric_cols if c in combined.columns]
    combined_summary = combined.groupby('model')[metric_cols].agg(['mean', 'std'])
    combined_summary.columns = [f'{col}_{stat}' for col, stat in combined_summary.columns]
    combined_summary = combined_summary.reset_index()
    combined_summary.to_csv(combined_summary_path, index=False)

    print('combined per-run saved:', combined_path)
    print('combined summary saved:', combined_summary_path)
    display(combined)
    display(combined_summary)
else:
    print('YOLO summary not found:', YOLO_SUMMARY_PER_RUN)
    print('Run the evaluation cell in shrimp.ipynb first if you want one combined table.')

YOLO summary not found: /content/drive/MyDrive/shrimp/test_summary_per_run.csv
Run the evaluation cell in shrimp.ipynb first if you want one combined table.


## Notes

- For the most defensible comparison, use the same Roboflow version for both `/content/shrimp-yolo` and `/content/shrimp-coco`.
- Faster R-CNN and EfficientDet use a YOLO-style letterbox resize so the input geometry is closer to Ultralytics validation at `imgsz=512`.
- Faster R-CNN and EfficientDet checkpoints are selected by validation `mAP50-95`; test results are computed only after selection.
- `BATCH_SIZE=4` with gradient accumulation is used to approximate YOLO's effective batch size of 32 on Colab GPUs. If your GPU can fit a larger batch, increase `BATCH_SIZE` and keep `EFFECTIVE_BATCH_SIZE=32`.
- If a model runs out of VRAM, reduce `BATCH_SIZE` to `2` or `1`; report that setting with the results.

In [19]:
from google.colab import runtime

print("Archive saved. Disconnecting runtime...")
runtime.unassign()

Archive saved. Disconnecting runtime...
